# PySpark intro

row object in pyspark

In [0]:
from pyspark.sql import Row

row1 = Row(name="Bella", animal_type="rabbit", age="4")
row1

In [0]:
row2 = Row(name="Charlie", animal_type="dog", age=5)
row2

create spark dataframe

can be created from
- Row
- list of dicts
- table
- csv
- parquet
- database
etc

In [0]:
df = spark.createDataFrame([row1, row2])
df

In [0]:
df.show()

In [0]:
df.take(1)

In [0]:
display(df.take(2))

In [0]:
display(df)

read from csv

In [0]:
from pathlib import Path

DATA_PATH = Path().resolve() / "data"

print(DATA_PATH)

In [0]:
df_athletes = spark.read.csv(str(DATA_PATH / "athlete_events.csv"), header=True)
display(df_athletes.take(5))

In [0]:
df_athletes.printSchema()

In [0]:
df_athletes.columns

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ByteType, ShortType

schema = StructType([
  StructField("ID", IntegerType(), True),
  StructField("Name", StringType(), True),
  StructField("Sex", StringType(), True),
  StructField("Age", ByteType(), True),
  StructField("Height", ShortType(), True),
  StructField("Weight", ShortType(), True),
  StructField("Team", StringType(), True),
  StructField("NOC", StringType(), True),
  StructField("Games", StringType(), True),
  StructField("Year", ShortType(), True),
  StructField("Season", StringType(), True),
  StructField("City", StringType(), True),
  StructField("Sport", StringType(), True),
  StructField("Event", StringType(), True),
  StructField("Medal", StringType(), True)
])
df_atheles_schema = spark.read.csv(str(DATA_PATH / "athlete_events.csv"), header=True, schema=schema)

display(df_atheles_schema.take(5))

EDA

In [0]:
from pyspark.sql.functions import col, sum

nulls = df_atheles_schema.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_atheles_schema.columns])

display(nulls)

In [0]:
display(df_atheles_schema.groupBy("NOC").count().filter("NOC = 'SWE'"))

Combining sql and dataframes

In [0]:
df_atheles_schema.createOrReplaceTempView("df_atheles_schema")

df_swe_medals = spark.sql("""
          SELECT 
            Sport,
            COUNT(Medal) as medals
          FROM df_atheles_schema
          WHERE NOC = 'SWE' AND Medal in ('Gold', 'Silver', 'Bronze')
          GROUP BY Sport
          ORDER BY medals DESC
""")

display(df_swe_medals)

In [0]:
fig = df_swe_medals.plot(kind="bar", x="medals", y="Sport")

fig.update_layout(yaxis = {"autorange": "reversed"})

SQL cells

In [0]:
%sql
FROM df_atheles_schema LIMIT 1

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS data;

CREATE SCHEMA IF NOT EXISTS data.olympics;

CREATE OR REPLACE TABLE data.olympics.sweden_medals AS
(
  SELECT
    Name,
    Age,
    Year,
    Sport,
    Medal
  FROM
    df_atheles_schema
  WHERE
    NOC = 'SWE' AND
    Medal IN ('Gold', 'Silver', 'Bronze')
);

FROM data.olympics.sweden_medals LIMIT 4